In [3]:
from functools import partial

import astroplan as ap
from astropy.coordinates import EarthLocation
import astropy.units as u

from astropaul.database import html_path
import astropaul.targetlistcreator as tlc
import astropaul.html as html
import astropaul.phase as ph

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
name = "Fan Mountain SIDE 2026 Fall"
html_dir = html_path() / name
html.clear_directory(html_dir)

session = tlc.ObservingSession(
    ap.Observer(EarthLocation(lat="37:52:41.35", lon="-78:41:34.92", height=566 * u.m), timezone="utc", name="Fan Mountain")
)

session.add_day_range("2026-08-29", "2026-12-31")

phase_event_defs = [
    ph.PhaseEventDef("Not in Eclipse", partial(ph.calc_time_of_gress, ingress=False)),
    ph.PhaseEventDef("Eclipse", partial(ph.calc_time_of_gress, ingress=True)),
]

fan_targets = [
    "TIC 63459761",
    "TIC 344541836",
    "TIC 418699570",
]

min_altitude = 35 * u.deg


all_targets = tlc.TargetList.load()

creator = tlc.TargetListCreator(name=name, phase_event_defs=phase_event_defs)
creator.steps = [
    partial(tlc.filter_targets, criteria=lambda df: (df["Target Name"].isin(fan_targets))),
    partial(
        tlc.add_observability,
        observing_session=session,
        calc_moon_distance=True,
        observability_threshold=(min_altitude, 80 * u.deg),
    ),
    partial(tlc.filter_targets, criteria=lambda df: df["Observable Any Night"]),
    partial(
        tlc.add_phase_events,
        observing_session=session,
        phase_event_defs=phase_event_defs,
        event_types=["Mid Eclipse", "Eclipse"],
    ),
]
tl = creator.calculate(initial_list=all_targets, verbose=False)

# html.render_observing_pages(tl, None, {}, html_dir)
print(tl.summarize())

/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  answer.target_list = answer.target_list.assign(**new_cols)
/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  answer.target_list = answer.target_list.assign(**new_cols)
/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: Perfor

Name: All Targets
Criteria:
  Target list loaded from file: TargetList_2026-08-15_09h57m58s.tl (734 targets)
  lambda df: (df["Target Name"].isin(['TIC 63459761', 'TIC 344541836', 'TIC 418699570'])) (3 targets)
  Observability calculated at Fan Mountain in 15.0 min intervals from 2026-08-28 to 2026-12-31
    AltitudeConstraint: {'min': np.float64(35.0), 'max': np.float64(80.0), 'boolean_constraint': True}
  lambda df: df["Observable Any Night"] (3 targets)
  
3 targets:
     3 QuadEB
Column Count (primary, secondary):
    Target: (3, 4)
    RV Calibration Targets: (1, 2)
    List: (0, 19)
    Count: (8, 0)
    TESS Data: (4, 0)
    Gaia Bailer Jones: (1, 2)
    Observable: (5, 500)
Associated tables:
    3248 rows,   3 columns: Catalog Membership
    1166 rows,   2 columns: List Memberships
     894 rows,   7 columns: Ephemerides
     716 rows, 126 columns: TESS
     308 rows, 105 columns: Gaia DR3
      49 rows,  16 columns: WDS
     258 rows,  10 columns: DSSI Observations
      44 r

In [5]:
html.render_observing_pages(tl, None, {}, html_dir)